<a href="https://colab.research.google.com/github/ho3eines/NotBookLM/blob/main/localAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Add cloudflare gpg key
!sudo mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-public-v2.gpg | sudo tee /usr/share/keyrings/cloudflare-public-v2.gpg >/dev/null

# Add this repo to your apt repositories
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-public-v2.gpg] https://pkg.cloudflare.com/cloudflared any main' | sudo tee /etc/apt/sources.list.d/cloudflared.list

# install cloudflared
!sudo apt-get update && sudo apt-get install cloudflared

deb [signed-by=/usr/share/keyrings/cloudflare-public-v2.gpg] https://pkg.cloudflare.com/cloudflared any main
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [97.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:12 https://pkg.cloudflare.com/

In [2]:
!pip install fastapi uvicorn transformers torch accelerate

In [3]:
%%writefile app.py
from fastapi import FastAPI
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

app = FastAPI()

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

@app.get("/chat")
def chat(q: str):
    inputs = tokenizer(q, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {"response": answer}


Writing app.py


In [13]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > local.log 2>&1 &

In [5]:
!sudo cloudflared service install eyJhIjoiYzViMTMwZDA4ODM3NDViNTk3NmY0MjNjOGFmNWFmMjUiLCJ0IjoiMmZkMjJlYzAtMTg2Yy00YTkyLThlNWMtZWMwM2JhZmRhMmJjIiwicyI6IlpqazJabVE1WkRJdE1HUmpNUzAwTkdOa0xXRmtPR1l0TjJSaFptVTJOVFV3Wm1ZMSJ9

2026-06-26T20:28:09Z INF Using SysV
2026-06-26T20:28:09Z INF Linux service for cloudflared installed successfully


In [11]:
!nohup cloudflared tunnel run --token "eyJhIjoiYzViMTMwZDA4ODM3NDViNTk3NmY0MjNjOGFmNWFmMjUiLCJ0IjoiMmZkMjJlYzAtMTg2Yy00YTkyLThlNWMtZWMwM2JhZmRhMmJjIiwicyI6IlpqazJabVE1WkRJdE1HUmpNUzAwTkdOa0xXRmtPR1l0TjJSaFptVTJOVFV3Wm1ZMSJ9" > cloudflared.log 2>&1 &

In [14]:
!curl "http://localhost:8000/chat" -G --data-urlencode "q=What is the capital of France?"

{"response":"What is the capital of France?\n\n# Answer\nThe capital of France is Paris."}

To verify the FastAPI server is running and accessible, first, identify the tunnel URL from the `nohup.out` file. Then, you can make a `curl` request to the `/chat` endpoint. Remember to replace `YOUR_TUNNEL_URL` with the actual URL you find in `nohup.out`.